In [ ]:
from config import TMP_DATA_DIR
from loaders.pd_loader import Registry
import polars as pl

registry = Registry(TMP_DATA_DIR)
registry.get_all_datasets()

In [ ]:
"""preprocessing steps
1. load as string (including file encoding)
2. identify schema from initial col audit
3. validate columns vs assigned schema
4. attempt dtype casting
5. column integrity check (detect/repair rows shifted by a missing field)
6. cross-column consistency check (start/end/duration)
7. null audit
8. value distribution and dupe checks
9. normalize station ids with current rt schema
"""
# load 2016 data as str and get file schema version
dat2016 = registry.get_dataset("2016")
dat2016.get_struct()

In [ ]:
# validate columns vs assigned schema
# backfill missing station_ids later
schema_report = pl.DataFrame([dat2016.validate_schema(f) for f in dat2016.get_fnames()])
schema_report

In [ ]:
# attempt dtype casting
casted = {}
cast_errors = {}
for f in dat2016.get_fnames():
    casted[f], cast_errors[f] = dat2016.cast_to_schema(f)
    print(f"{f}: {len(casted[f])} rows cast, {len(cast_errors[f])} cast failures")

casted["tor-trips-2016-Q3-.csv"].schema

In [ ]:
# column integrity check, find rows where raw field is missing entirely
for f in dat2016.get_fnames():
    suspect = dat2016.check_column_integrity(f)
    print(f"{f}: {len(suspect)} rows with suspected column shift")

In [ ]:
# cross-column consistency check (start/end/duration)
from loaders.pd_loader import check_consistency

consistency_report = []
anomalies_by_file = {}
for f, df in casted.items():
    anomalies_by_file[f] = check_consistency(df)
    consistency_report.append({
        "file": f,
        "n_rows": len(df),
        "n_anomalies": len(anomalies_by_file[f]),
        "pct_anomalies": len(anomalies_by_file[f]) / len(df),
    })

pl.DataFrame(consistency_report)

In [ ]:
# seems like midnight trips are the main source, where end_time's month changes rather than day
# (e.g. start 2016-01-10 22:01 -> end saved as 2016-02-10 00:18, but intended end is 2016-01-11 00:18)
anomalies_by_file["tor-trips-2016-Q4.csv"][
    ["start_time", "end_time", "trip_duration", "computed_duration", "provided_duration"]
].head(10)

In [ ]:
dat2016.compare_dfs(func=lambda df: df.null_count().row(0, named=True))

In [ ]:
# missing station names here has no way to recover record data
anomalies = []
for f in dat2016.get_fpaths():
    df = dat2016.get_df(f)
    missing = df.filter(pl.col("start_station_name").is_null() | pl.col("end_station_name").is_null())

    if not missing.is_empty():
        anomalies.append(missing.with_columns(pl.lit(f).alias("file")))

anomalies2016 = pl.concat(anomalies, how="diagonal_relaxed")
anomalies2016 = anomalies2016.select(["file"] + [c for c in anomalies2016.columns if c != "file"])
anomalies2016

In [ ]:
# value distribution and dupe checks
for f in dat2016.get_fnames():
    dupes = dat2016.check_duplicates(f)
    print(f"{f}: {len(dupes)} duplicate trip_id rows")

dat2016.value_distribution("tor-trips-2016-Q3-.csv", "user_type")

In [ ]:
# standardize station ids with current rt schema by crosswalking against historical data and the live feed
from validation.crosswalk import get_rt_stations, build_rt_lookup, match_station_names, match_station_names_historical
from loaders.pd_loader import build_historical_lookup
from config import GBFS_FEED

historical_lookup_2016 = build_historical_lookup(registry, "2016")
print(f"historical lookup (2014-2015 + 2017): {len(historical_lookup_2016)} stations")

rt_stations = get_rt_stations(GBFS_FEED)
rt_lookup = build_rt_lookup(rt_stations)
len(rt_lookup)

In [ ]:
# apply the matcher to each casted file: historical tier first, RT tier (unresolved)
matched = {
    f: match_station_names(match_station_names_historical(df, historical_lookup_2016), rt_lookup)
    for f, df in casted.items()
}

match_summary = []
for f, df in matched.items():
    for prefix in ("start", "end"):
        via = df[f"{prefix}_matched_via"].value_counts()
        counts = dict(zip(via[f"{prefix}_matched_via"], via["count"]))
        historical_exact = counts.get("historical_exact_name", 0)
        historical_fuzzy = counts.get("historical_fuzzy_name", 0)
        rt_exact = counts.get("exact_name", 0)
        rt_fuzzy = counts.get("fuzzy_name", 0)
        match_summary.append({
            "file": f,
            "endpoint": prefix,
            "historical_exact": historical_exact,
            "historical_fuzzy": historical_fuzzy,
            "rt_exact": rt_exact,
            "rt_fuzzy": rt_fuzzy,
            "unmatched": len(df) - historical_exact - historical_fuzzy - rt_exact - rt_fuzzy,
            "n_rows": len(df),
        })

pl.DataFrame(match_summary)

In [ ]:
dat2016_consolidated = dat2016.consol(rt_lookup=rt_lookup, historical_lookup=historical_lookup_2016)
print(f"consolidated rows: {len(dat2016_consolidated)}")
print(f"start_id_inferred rate: {dat2016_consolidated['start_id_inferred'].mean():.1%}")
print(f"end_id_inferred rate:   {dat2016_consolidated['end_id_inferred'].mean():.1%}")
print(f"start historical-matched share: {dat2016_consolidated['start_matched_via'].fill_null('').str.starts_with('historical_').mean():.1%}")
print(f"end historical-matched share:   {dat2016_consolidated['end_matched_via'].fill_null('').str.starts_with('historical_').mean():.1%}")
dat2016_consolidated.head()

In [ ]:
# run pipeline for 2017-2026
years_2017_2026 = [str(y) for y in range(2017, 2027)]

consolidated_by_year = {}
year_summary = []

for year in years_2017_2026:
    entry = registry.get_dataset(year)
    historical_lookup = build_historical_lookup(registry, year)
    consolidated = entry.consol(rt_lookup=rt_lookup, historical_lookup=historical_lookup)
    consolidated_by_year[year] = consolidated

    counts_df = entry.get_anomalies()["anomaly_type"].value_counts()
    anomaly_counts = dict(zip(counts_df["anomaly_type"], counts_df["count"]))

    if consolidated.is_empty():
        year_summary.append({"year": year, "n_files": len(entry.get_fnames()), "n_rows": 0})
        continue

    year_summary.append({
        "year": year,
        "n_files": len(entry.get_fnames()),
        "n_rows": len(consolidated),
        "column_shift_repaired": anomaly_counts.get("column_shift", 0),
        "cast_errors": anomaly_counts.get("cast_error", 0),
        "consistency_anomalies": anomaly_counts.get("consistency", 0),
        "pct_consistency_anomalies": anomaly_counts.get("consistency", 0) / len(consolidated),
        "duplicate_trip_ids": anomaly_counts.get("duplicate_trip_id", 0),
        "start_id_inferred_rate": consolidated["start_id_inferred"].mean(),
        "end_id_inferred_rate": consolidated["end_id_inferred"].mean(),
        "start_historical_matched_rate": consolidated["start_matched_via"].fill_null("").str.starts_with("historical_").mean(),
        "end_historical_matched_rate": consolidated["end_matched_via"].fill_null("").str.starts_with("historical_").mean(),
    })

year_summary_df = pl.DataFrame(year_summary)
year_summary_df

In [ ]:
# yearly summary
full_summary = pl.concat([
    pl.DataFrame([{
        "year": "2016",
        "n_files": len(dat2016.get_fnames()),
        "n_rows": len(dat2016_consolidated),
        "column_shift_repaired": 0,
        "cast_errors": sum(len(e) for e in dat2016.get_cast_errors().values()),
        "consistency_anomalies": sum(len(a) for a in anomalies_by_file.values()),
        "pct_consistency_anomalies": sum(len(a) for a in anomalies_by_file.values()) / len(dat2016_consolidated),
        "duplicate_trip_ids": sum(len(dat2016.check_duplicates(f)) for f in dat2016.get_fnames()),
        "start_id_inferred_rate": dat2016_consolidated["start_id_inferred"].mean(),
        "end_id_inferred_rate": dat2016_consolidated["end_id_inferred"].mean(),
        "start_historical_matched_rate": dat2016_consolidated["start_matched_via"].fill_null("").str.starts_with("historical_").mean(),
        "end_historical_matched_rate": dat2016_consolidated["end_matched_via"].fill_null("").str.starts_with("historical_").mean(),
    }]),
    year_summary_df,
], how="diagonal_relaxed")

full_summary

In [ ]:
anomaly_frames = [dat2016.get_anomalies().with_columns(pl.lit("2016").alias("year"))]
for year in years_2017_2026:
    anomaly_frames.append(registry.get_dataset(year).get_anomalies().with_columns(pl.lit(year).alias("year")))

anomaly_df = pl.concat(anomaly_frames, how="diagonal_relaxed").select(["year", "file", "row", "trip_id", "anomaly_type", "detail"])

print(f"total anomaly records: {len(anomaly_df)}")
anomaly_df["anomaly_type"].value_counts()

In [ ]:
anomaly_df.filter(pl.col("anomaly_type") == "column_shift").head()

In [ ]:
anomaly_df.filter(pl.col("anomaly_type") == "cast_error").head()